# ML-1M: corrected SparseWalker + one causal attention layer

This is a **diagnostic control**, not the final architecture. It trains from scratch:

`corrected Walker v1.1 -> 1 causal self-attention block -> residual FFN -> tied item scorer`

At every validation checkpoint it evaluates the **same trained weights** with attention **ON** and **OFF**. A large `attention_contribution` means temporal retrieval is rescuing information that the Walker recurrence alone cannot preserve.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess, torch
REPO='/content/Sparsewalker'
BRANCH='agent/walker-attention-control'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
SRC=f'{REPO}/src'
sys.path.insert(0,SRC)
for name in list(sys.modules):
    if name=='sparsewalker' or name.startswith('sparsewalker.'):
        del sys.modules[name]
assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU',torch.cuda.get_device_name(0),'bf16',torch.cuda.is_bf16_supported())
print('BRANCH',BRANCH)


## Run

The runner first executes a **causal leak test**. Training uses the canonical ML-1M split, max_len=200, FullCE, corrected v1.1 Walker recurrence, K=8, degree=4, two graph hops, BF16 training, and **no pursuit**.

Watch `val_on_NDCG@10`, `val_off_NDCG@10`, `attention_contribution`, and `eval_cost_ratio`.

In [ ]:
import runpy, sys
SCRIPT=f'{REPO}/experiments/run_ml1m_walker_attention_control.py'
sys.argv=[
    SCRIPT,
    '--seed','42',
    '--max-epochs','50',
    '--eval-every','5',
    '--patience','20',
    '--batch-size','128',
    '--eval-batch-size','1024',
    '--attn-heads','2',
    '--ff-mult','4',
    '--dropout','0.1',
]
print('INPROCESS WALKER+ATTENTION START',flush=True)
runpy.run_path(SCRIPT,run_name='__main__')
print('INPROCESS WALKER+ATTENTION END',flush=True)


## Inspect result

Paste the epoch-5 and epoch-10 `EVAL` lines back into ChatGPT first.

In [ ]:
import json, pandas as pd
from pathlib import Path
root=Path('/content/drive/MyDrive/sparsewalker_attention_control/ml1m/seed42')
hp=root/'history.csv'
rp=root/'result.json'
if hp.exists(): display(pd.read_csv(hp))
if rp.exists(): print(json.dumps(json.loads(rp.read_text()),indent=2))
